# P4 spike: grammar-constrained decoding inside the agent loop, small local model

**Question:** can a small local model (Qwen3-8B, 4-bit AWQ, thinking off) run our agent loop at all, and does the graph-derived grammar cut invalid and made-up queries?

Two runs on the clinic **train** questions (16 Q), both arms, judged by GLM-4.7 on OpenRouter:
- **A** `--grammar none`: the model writes freely.
- **B** `--grammar cfg`: every query-writing reply must match the grammar (labels, relationship types and property keys of the loaded graph only; read-only Cypher; one line).

1. **Runtime:** GPU, **L4** (Colab Pro). A T4 also works for the 8B model, just slower.
2. **Colab Secrets** (key icon): `OPENROUTER_API_KEY` with *Notebook access* on (the judge). `HF_TOKEN` is optional.
3. **Runtime → Run all.** Approve the Drive prompt with the account that holds `MyDrive/chatdt` (the snapshots are already there).

Expect ~10 min setup (vLLM install + 6 GB model download), then roughly 20–40 min per run. Judge cost < $0.20. Results resume across sessions like the other run notebooks.

In [ ]:
# 1. Code + secrets
import os, subprocess
from google.colab import userdata

BRANCH = 'journal/agentic-cfg'
REPO = 'github.com/IvansSmirnoff/Chat.DT.git'

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.isdir('/content/Chat.DT'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, '/content/Chat.DT'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Chat.DT', 'pull', '-q', '--ff-only'], check=True)
os.chdir('/content/Chat.DT')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

os.environ['OPENROUTER_API_KEY'] = secret('OPENROUTER_API_KEY') or ''
assert os.environ['OPENROUTER_API_KEY'], 'Add OPENROUTER_API_KEY to Colab Secrets'
os.environ.update({
    'NEO4J_URI': 'bolt://127.0.0.1:7687', 'NEO4J_USER': 'neo4j', 'NEO4J_PASSWORD': 'password',
    'IFCBENCH_MANIFEST_PATH': '/content/Chat.DT/data/ifcbench/manifest.yaml',
})

In [ ]:
# 2. GPU + dependencies + Neo4j. vLLM goes in after the base requirements, so its own pins win.
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
assert gpu, 'No GPU: Runtime -> Change runtime type -> L4 (or T4)'
driver_major = int(gpu.split(',')[1].strip().split('.')[0])
print('GPU:', gpu)

!pip install -q -r requirements-base.txt uv
VLLM = '0.32.0'
if driver_major >= 580:   # CUDA 13 builds need an R580+ driver
    !uv pip install --system -q vllm=={VLLM} --torch-backend=auto
else:                     # older driver: the CUDA 12.9 build from the release page
    !uv pip install --system -q "https://github.com/vllm-project/vllm/releases/download/v{VLLM}/vllm-{VLLM}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129 --index-strategy unsafe-best-match
# Colab's preinstalled torchaudio is built for Colab's torch; vLLM brings a newer torch, and transformers
# imports any torchaudio it finds, which then refuses the CUDA mismatch. Nothing here needs audio.
!pip uninstall -y -q torchaudio
!vllm download-kernels || echo '(kernel download skipped)'
subprocess.run(['vllm', '--version'], check=True)   # same imports as `vllm serve`: fail here, not in the background
!python -c "import torch; print('torch', torch.__version__, '| cuda ok:', torch.cuda.is_available())"

import psutil
ram_gb = psutil.virtual_memory().total / 2**30
os.environ['NEO4J_HEAP_MAX'] = '16g' if ram_gb > 60 else '4g'
os.environ['NEO4J_PAGECACHE'] = '8g' if ram_gb > 60 else '2g'
!bash scripts/colab/bootstrap.sh

In [ ]:
# 3. Persist snapshots + results on Drive (resume across sessions)
from google.colab import drive
drive.mount('/content/drive')
PERSIST = '/content/drive/MyDrive/chatdt'
for local, remote in [('data/ifcbench/cache', 'ifcbench/cache'),
                      ('results/ifcbench/runs', 'ifcbench/runs')]:
    os.makedirs(f'{PERSIST}/{remote}', exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if not os.path.islink(local):
        subprocess.run(['rm', '-rf', local])
        os.symlink(f'{PERSIST}/{remote}', local)
    print(local, '->', os.readlink(local))

In [ ]:
# 4. IFC files from HuggingFace (pinned dataset commit, sha256-verified).
#    --project dental_clinic for Tier A; --split test for all 31 test-split files (1.34 GB).
!python scripts/ifcbench/fetch.py --project dental_clinic

In [ ]:
# 5. Serve the local model with vLLM (background) and wait until it answers.
#    --structured-outputs-config.backend guidance: llguidance, the Lark dialect our grammar is written in.
#    No --reasoning-parser: thinking is switched off per request, so the grammar covers the whole reply.
import time, urllib.request
MODEL = 'Qwen/Qwen3-8B-AWQ'

def start_vllm(model):
    subprocess.run(['pkill', '-f', 'vllm serve'], check=False)
    time.sleep(5)
    log = open('/content/vllm.log', 'w')
    proc = subprocess.Popen(
        ['vllm', 'serve', model, '--dtype', 'half', '--max-model-len', '32768',
         '--gpu-memory-utilization', '0.90', '--max-num-seqs', '8', '--port', '8000',
         '--structured-outputs-config.backend', 'guidance'],
        stdout=log, stderr=subprocess.STDOUT)
    for i in range(180):                       # model download + load + compile: up to ~15 min
        if proc.poll() is not None:
            break
        try:
            urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2)
            print(f'{model} up after {i * 5} s')
            return proc
        except Exception:
            time.sleep(5)
    print(open('/content/vllm.log').read()[-4000:])
    raise RuntimeError('vLLM did not come up; log tail above')

vllm_proc = start_vllm(MODEL)
os.environ['VLLM_BASE_URL'] = 'http://127.0.0.1:8000/v1'

In [ ]:
# 6. Check that the grammar is really applied, before spending a run on it.
#    (vLLM silently ignores the old guided_* fields, so test the real one: the reply must obey it,
#    and a broken grammar must be refused.)
import json, httpx
def ask(body):
    return httpx.post('http://127.0.0.1:8000/v1/chat/completions', timeout=120, json={
        'model': MODEL, 'temperature': 0.1, 'max_tokens': 20,
        'chat_template_kwargs': {'enable_thinking': False}, **body})

r = ask({'messages': [{'role': 'user', 'content': 'Write a short poem about doors.'}],
         'structured_outputs': {'grammar': 'start: "yes" | "no"'}})
reply = r.json()['choices'][0]['message']['content']
print('constrained reply:', repr(reply))
assert reply.strip() in ('yes', 'no'), 'grammar NOT applied: stop here'

bad = ask({'messages': [{'role': 'user', 'content': 'x'}], 'structured_outputs': {'grammar': 'start: undefined_rule'}})
print('bad grammar ->', bad.status_code)
assert bad.status_code >= 400, 'an invalid grammar was accepted: the field is being ignored'
print('OK: grammars are enforced')

In [ ]:
# 7. Runs A (free) and B (grammar). Same settings otherwise; each run resumes if re-run.
TAG = MODEL.split('/')[-1].lower().replace('-awq', '').replace('.', '')   # qwen3-8b
COMMON = (f'--model-id dental_clinic__arc --model-id dental_clinic__str --split train '
          f'--llm vllm:{MODEL} --reasoning off --judge-provider z-ai --workers 8')
RUN_A = f'p4_{TAG}_nothink_none'
RUN_B = f'p4_{TAG}_nothink_cfg'
!python -m src.agent.run {COMMON} --grammar none --run {RUN_A}
!python -m src.agent.run {COMMON} --grammar cfg --run {RUN_B}

In [ ]:
# 8. Compare A vs B, and the gate.
import pandas as pd
def summary(run):
    return json.load(open(f'results/ifcbench/runs/{run}/summary.json'))

rows = []
for run, label in [(RUN_A, 'A free'), (RUN_B, 'B grammar')]:
    for paradigm, c in summary(run).items():
        rows.append({'run': label, 'arm': paradigm, 'correct': f"{c['correct']}/{c['n']}",
                     'abstained': c['abstained'], 'queries/q': c['mean_queries'],
                     'query errors': c.get('query_error_rate'), 'step limit': c['step_limit'],
                     'parse errors': c['parse_error'], 'token limit': c['token_limit'],
                     's/q': c['mean_wall_s']})
display(pd.DataFrame(rows))

# How many of A's error-free queries name something the graph does not have (what B forbids)?
!python scripts/grammar_coverage.py results/ifcbench/runs/{RUN_A}/episodes.jsonl --manifest data/ifcbench/manifest.yaml | head -3

agentic_a = summary(RUN_A)['agentic']
print()
print('GATE: agentic A correct', agentic_a['correct'], '/', agentic_a['n'],
      '| query error rate', agentic_a.get('query_error_rate'))
print('-> <= 2/16 with few invalid or made-up queries: too weak to show a grammar effect; try the 14B below.')
print('-> many invalid or made-up queries in A: B says whether the grammar fixes them.')

## Optional: second model size (L4 only)

Qwen3-14B-AWQ (~10 GB) does not fit a T4 with a 32k context. Set `RUN_14B = True` and run the cell: it restarts vLLM with the bigger model and repeats A and B.

In [ ]:
RUN_14B = False
if RUN_14B:
    MODEL = 'Qwen/Qwen3-14B-AWQ'
    vllm_proc = start_vllm(MODEL)
    TAG = 'qwen3-14b'
    COMMON = (f'--model-id dental_clinic__arc --model-id dental_clinic__str --split train '
              f'--llm vllm:{MODEL} --reasoning off --judge-provider z-ai --workers 8')
    RUN_A, RUN_B = f'p4_{TAG}_nothink_none', f'p4_{TAG}_nothink_cfg'
    !python -m src.agent.run {COMMON} --grammar none --run {RUN_A}
    !python -m src.agent.run {COMMON} --grammar cfg --run {RUN_B}
    for run in (RUN_A, RUN_B):
        print(run, {p: f"{c['correct']}/{c['n']}" for p, c in summary(run).items()})